# Fire Scenario Analysis (SCN003)

![Fire Scenario Petri Net](Images/Fire%20petrinet.png)

This notebook analyzes the impact of a warehouse fire scenario on the supply chain.

**Parameters (copy/paste to other notebooks):**
- `ISSUE_DATE`: Reference date for analysis (YYYYMMDD, blank = from scenario)
- `FUTURE_TIME_HORIZON`: Days to look ahead (default: 90)

**Analysis Sections:**
1. Customer orders due to ship from fire location
2. Current inventory across the supply chain
3. All customer orders (with customer locations)
4. Incoming supply - goods receipts from production
5. Lost inventory at fire location
6. Product shortfall analysis (available vs. demand)

In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS
# =============================================================================
# These parameters can be copy/pasted to other notebooks

# --- Database Connection ---
dbutils.widgets.text("CATALOG", "sample_synthetic_sap", "Catalog Name")
dbutils.widgets.text("SCHEMA", "sap", "Schema Name")

# --- Analysis Parameters ---
dbutils.widgets.text("ISSUE_DATE", "", "Issue Date (YYYYMMDD, blank=from scenario)")
dbutils.widgets.text("FUTURE_TIME_HORIZON", "90", "Future Time Horizon (days)")

# --- Get Parameter Values ---
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
ISSUE_DATE_PARAM = dbutils.widgets.get("ISSUE_DATE")
FUTURE_TIME_HORIZON = int(dbutils.widgets.get("FUTURE_TIME_HORIZON"))

import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"Database:              {CATALOG}.{SCHEMA}")
print(f"Issue Date:            {ISSUE_DATE_PARAM or '(from scenario)'}")
print(f"Time Horizon:          {FUTURE_TIME_HORIZON} days")
print("=" * 70)

In [ ]:
# Load Fire Scenario Details and Apply Parameters
fire_info = spark.sql(f"""
    SELECT 
        scenario_id,
        description,
        plant,
        storage_loc,
        quantity as total_qty_scrapped,
        downtime_days,
        recovery_date,
        injected_at
    FROM {CATALOG}.{SCHEMA}.scenario_metadata
    WHERE scenario_id = 'SCN003'
""").collect()

if len(fire_info) > 0:
    fire = fire_info[0]
    
    # Get values from scenario
    FIRE_PLANT = fire['plant']
    FIRE_SLOC = fire['storage_loc']
    DOWNTIME_DAYS = fire['downtime_days'] or 30
    
    # Calculate date range using parameters
    if ISSUE_DATE_PARAM:
        # Use provided issue date
        issue_date_dt = datetime.strptime(ISSUE_DATE_PARAM, '%Y%m%d')
    else:
        # Calculate from scenario (fire date = recovery date - downtime)
        recovery_date = fire['recovery_date'][:8] if fire['recovery_date'] else '20250615'
        issue_date_dt = datetime.strptime(recovery_date, '%Y%m%d') - timedelta(days=DOWNTIME_DAYS)
    
    # Calculate end date based on time horizon
    end_date_dt = issue_date_dt + timedelta(days=FUTURE_TIME_HORIZON)
    
    # String formats for SQL queries
    ISSUE_DATE_STR = issue_date_dt.strftime('%Y%m%d')
    END_DATE_STR = end_date_dt.strftime('%Y%m%d')
    
    # For backward compatibility with existing cells
    FIRE_DATE_STR = ISSUE_DATE_STR
    
    print("=" * 70)
    print("FIRE SCENARIO DETAILS")
    print("=" * 70)
    print(f"Plant:              {FIRE_PLANT}")
    print(f"Storage Location:   {FIRE_SLOC}")
    print(f"Issue Date:         {ISSUE_DATE_STR}")
    print(f"Downtime:           {DOWNTIME_DAYS} days")
    print(f"Analysis Period:    {ISSUE_DATE_STR} to {END_DATE_STR} ({FUTURE_TIME_HORIZON} days)")
    print("=" * 70)
else:
    print("ERROR: No fire scenario (SCN003) found. Run the pipeline with SCN003 enabled.")
    dbutils.notebook.exit("No fire scenario found")

---
## 1. Customer Orders Due to Ship from Fire Location

Orders that were scheduled to ship from the fire-affected plant in the next 3 months but have not yet been delivered.

In [ ]:
# Customer orders due to ship from fire location (not yet shipped, next 3 months)
orders_from_fire_location = spark.sql(f"""
    WITH delivered_orders AS (
        SELECT DISTINCT f.VBELN
        FROM {CATALOG}.{SCHEMA}.vbfa f
        INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
    )
    SELECT 
        o.VBELN as Order_Number,
        o.KUNNR as Customer_ID,
        c.NAME1 as Customer_Name,
        c.ORT01 as Customer_City,
        c.LAND1 as Customer_Country,
        p.MATNR as Material,
        t.MAKTX as Material_Description,
        p.WERKS as Ship_From_Plant,
        CAST(p.KWMENG AS DOUBLE) as Order_Qty,
        CAST(p.NETWR AS DOUBLE) as Line_Value,
        e.EDATU as Requested_Delivery_Date,
        o.ERDAT as Order_Date
    FROM {CATALOG}.{SCHEMA}.vbak o
    INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
    LEFT JOIN {CATALOG}.{SCHEMA}.vbep e ON p.VBELN = e.VBELN AND p.POSNR = e.POSNR
    LEFT JOIN {CATALOG}.{SCHEMA}.kna1 c ON o.KUNNR = c.KUNNR
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
    WHERE p.WERKS = '{FIRE_PLANT}'
    AND o.VBELN NOT IN (SELECT VBELN FROM delivered_orders)
    AND e.EDATU >= '{FIRE_DATE_STR}'
    AND e.EDATU <= '{END_DATE_STR}'
    ORDER BY e.EDATU, o.VBELN
""")

df_fire_orders = orders_from_fire_location.toPandas()

print(f"Orders due to ship from Plant {FIRE_PLANT} (not yet shipped):")
print(f"  Total orders: {df_fire_orders['Order_Number'].nunique()}")
print(f"  Total lines:  {len(df_fire_orders)}")
print(f"  Total value:  £{df_fire_orders['Line_Value'].sum():,.2f}")
print(f"  Total qty:    {df_fire_orders['Order_Qty'].sum():,.0f} units")
print()
display(orders_from_fire_location)

---
## 2. Current Inventory in the Supply Chain

All inventory across all plants and storage locations.

In [ ]:
# Current inventory across the supply chain
current_inventory = spark.sql(f"""
    SELECT 
        d.WERKS as Plant,
        l.CITY as Plant_Name,
        d.LGORT as Storage_Location,
        d.MATNR as Material,
        t.MAKTX as Material_Description,
        a.MTART as Material_Type,
        CAST(d.LABST AS DOUBLE) as Available_Stock,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(d.LABST AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Stock_Value,
        CASE WHEN d.WERKS = '{FIRE_PLANT}' THEN 'FIRE LOCATION' ELSE '' END as Fire_Flag
    FROM {CATALOG}.{SCHEMA}.mard d
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON d.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON d.MATNR = a.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON d.MATNR = b.MATNR AND d.WERKS = b.BWKEY
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l ON d.WERKS = l.LOCNO
    WHERE CAST(d.LABST AS DOUBLE) > 0
    ORDER BY d.WERKS, d.LGORT, d.MATNR
""")

df_inventory = current_inventory.toPandas()

print("CURRENT INVENTORY ACROSS SUPPLY CHAIN")
print("=" * 70)

# Summary by plant
plant_summary = df_inventory.groupby(['Plant', 'Plant_Name']).agg({
    'Material': 'nunique',
    'Available_Stock': 'sum',
    'Stock_Value': 'sum'
}).reset_index()
plant_summary.columns = ['Plant', 'Plant_Name', 'Materials', 'Total_Stock', 'Total_Value']

for _, row in plant_summary.iterrows():
    fire_flag = " ** FIRE LOCATION **" if row['Plant'] == FIRE_PLANT else ""
    print(f"Plant {row['Plant']} ({row['Plant_Name']}){fire_flag}")
    print(f"  Materials: {row['Materials']}  |  Stock: {row['Total_Stock']:,.0f} units  |  Value: £{row['Total_Value']:,.2f}")

print("=" * 70)
print(f"TOTAL: {df_inventory['Available_Stock'].sum():,.0f} units  |  £{df_inventory['Stock_Value'].sum():,.2f}")
print()
display(current_inventory)

---
## 3. All Customer Orders for Next 3 Months

Complete view of customer demand with customer locations.

In [ ]:
# All customer orders for next 3 months with customer location
all_customer_orders = spark.sql(f"""
    WITH delivered_orders AS (
        SELECT DISTINCT f.VBELN
        FROM {CATALOG}.{SCHEMA}.vbfa f
        INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
    )
    SELECT 
        o.VBELN as Order_Number,
        o.KUNNR as Customer_ID,
        c.NAME1 as Customer_Name,
        c.ORT01 as Customer_City,
        c.REGIO as Customer_Region,
        c.LAND1 as Customer_Country,
        p.MATNR as Material,
        t.MAKTX as Material_Description,
        p.WERKS as Supplying_Plant,
        CAST(p.KWMENG AS DOUBLE) as Order_Qty,
        CAST(p.NETWR AS DOUBLE) as Line_Value,
        e.EDATU as Requested_Delivery_Date,
        CASE 
            WHEN o.VBELN IN (SELECT VBELN FROM delivered_orders) THEN 'Delivered'
            ELSE 'Open'
        END as Order_Status,
        CASE WHEN p.WERKS = '{FIRE_PLANT}' THEN 'Yes' ELSE 'No' END as From_Fire_Plant
    FROM {CATALOG}.{SCHEMA}.vbak o
    INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
    LEFT JOIN {CATALOG}.{SCHEMA}.vbep e ON p.VBELN = e.VBELN AND p.POSNR = e.POSNR
    LEFT JOIN {CATALOG}.{SCHEMA}.kna1 c ON o.KUNNR = c.KUNNR
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON p.MATNR = t.MATNR
    WHERE e.EDATU >= '{FIRE_DATE_STR}'
    AND e.EDATU <= '{END_DATE_STR}'
    ORDER BY e.EDATU, o.VBELN
""")

df_all_orders = all_customer_orders.toPandas()

print("ALL CUSTOMER ORDERS FOR NEXT 3 MONTHS")
print("=" * 70)

# Summary
open_orders = df_all_orders[df_all_orders['Order_Status'] == 'Open']
fire_plant_orders = open_orders[open_orders['From_Fire_Plant'] == 'Yes']

print(f"Total order lines:        {len(df_all_orders)}")
print(f"  - Delivered:            {len(df_all_orders) - len(open_orders)}")
print(f"  - Open:                 {len(open_orders)}")
print(f"  - From fire plant:      {len(fire_plant_orders)} (£{fire_plant_orders['Line_Value'].sum():,.2f})")
print(f"Total order value:        £{df_all_orders['Line_Value'].sum():,.2f}")
print(f"Open order value:         £{open_orders['Line_Value'].sum():,.2f}")
print()

# By customer country
print("Open Orders by Customer Country:")
country_summary = open_orders.groupby('Customer_Country').agg({
    'Order_Number': 'nunique',
    'Order_Qty': 'sum',
    'Line_Value': 'sum'
}).reset_index().sort_values('Line_Value', ascending=False)
print(country_summary.to_string(index=False))
print()
display(all_customer_orders)

---
## 4. Incoming Supply (Goods Receipts from Production)

Material scheduled to arrive into the supply chain - quantities, locations, and availability dates.

In [ ]:
# Incoming Supply: Goods Receipts from Production (Movement Type 101)
# Shows material being added to inventory - quantities, locations, and dates
incoming_supply = spark.sql(f"""
    SELECT 
        m.MBLNR as Document_Number,
        m.MATNR as Material,
        t.MAKTX as Material_Description,
        a.MTART as Material_Type,
        m.WERKS as Plant,
        l.CITY as Plant_Name,
        m.LGORT as Storage_Location,
        CAST(m.MENGE AS DOUBLE) as Quantity,
        m.MEINS as Unit,
        m.BUDAT as Posting_Date,
        m.CPUDT as Entry_Date,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Receipt_Value,
        CASE WHEN m.WERKS = '{FIRE_PLANT}' THEN 'Yes' ELSE 'No' END as At_Fire_Plant
    FROM {CATALOG}.{SCHEMA}.matdoc m
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON m.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON m.MATNR = a.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON m.MATNR = b.MATNR AND m.WERKS = b.BWKEY
    LEFT JOIN {CATALOG}.{SCHEMA}.sapapo_loc l ON m.WERKS = l.LOCNO
    WHERE m.BWART = '101'
    AND m.BUDAT >= '{ISSUE_DATE_STR}'
    AND m.BUDAT <= '{END_DATE_STR}'
    AND m.MBLNR NOT LIKE 'SCN%'
    ORDER BY m.BUDAT, m.WERKS, m.MATNR
""")

df_incoming = incoming_supply.toPandas()

print(f"INCOMING SUPPLY - GOODS RECEIPTS FROM PRODUCTION")
print(f"Period: {ISSUE_DATE_STR} to {END_DATE_STR} ({FUTURE_TIME_HORIZON} days)")
print("=" * 70)

if len(df_incoming) > 0:
    # Summary by plant
    plant_summary = df_incoming.groupby(['Plant', 'Plant_Name']).agg({
        'Document_Number': 'count',
        'Material': 'nunique',
        'Quantity': 'sum',
        'Receipt_Value': 'sum'
    }).reset_index()
    plant_summary.columns = ['Plant', 'Plant_Name', 'Receipts', 'Materials', 'Total_Qty', 'Total_Value']
    
    for _, row in plant_summary.iterrows():
        fire_flag = " ** FIRE LOCATION **" if row['Plant'] == FIRE_PLANT else ""
        print(f"Plant {row['Plant']} ({row['Plant_Name']}){fire_flag}")
        print(f"  Receipts: {row['Receipts']}  |  Materials: {row['Materials']}  |  Qty: {row['Total_Qty']:,.0f}  |  Value: £{row['Total_Value']:,.2f}")
    
    print("=" * 70)
    print(f"TOTAL: {len(df_incoming)} receipts  |  {df_incoming['Quantity'].sum():,.0f} units  |  £{df_incoming['Receipt_Value'].sum():,.2f}")
    
    # Summary by date (weekly buckets)
    df_incoming['Week'] = pd.to_datetime(df_incoming['Posting_Date'], format='%Y%m%d').dt.to_period('W')
    weekly = df_incoming.groupby('Week').agg({
        'Quantity': 'sum',
        'Receipt_Value': 'sum'
    }).reset_index()
    
    print("\nWeekly Incoming Supply:")
    print("-" * 50)
    for _, row in weekly.iterrows():
        print(f"  {row['Week']}: {row['Quantity']:>12,.0f} units  |  £{row['Receipt_Value']:>12,.2f}")
    
    # Supply at fire plant
    fire_supply = df_incoming[df_incoming['At_Fire_Plant'] == 'Yes']
    if len(fire_supply) > 0:
        print(f"\n** Supply scheduled for FIRE PLANT ({FIRE_PLANT}): {fire_supply['Quantity'].sum():,.0f} units **")
else:
    print("No goods receipts found in the date range.")

print()
display(incoming_supply)

---
## 5. Lost Inventory at Fire Location

Detailed breakdown of inventory destroyed in the fire.

In [ ]:
# Lost inventory at fire location
lost_inventory = spark.sql(f"""
    SELECT 
        m.MATNR as Material,
        t.MAKTX as Material_Description,
        a.MTART as Material_Type,
        m.WERKS as Plant,
        m.LGORT as Storage_Location,
        CAST(m.MENGE AS DOUBLE) as Quantity_Lost,
        COALESCE(b.STPRS, 0) as Standard_Price,
        COALESCE(b.PEINH, 1) as Price_Unit,
        ROUND(CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1), 2) as Value_Lost
    FROM {CATALOG}.{SCHEMA}.matdoc m
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON m.MATNR = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON m.MATNR = a.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mbew b ON m.MATNR = b.MATNR AND m.WERKS = b.BWKEY
    WHERE m.MBLNR LIKE 'SCN003%'
    AND m.BWART = '551'
    ORDER BY CAST(m.MENGE AS DOUBLE) * COALESCE(b.STPRS, 0) / COALESCE(b.PEINH, 1) DESC
""")

df_lost = lost_inventory.toPandas()

print("LOST INVENTORY AT FIRE LOCATION")
print("=" * 70)

if len(df_lost) > 0:
    total_qty = df_lost['Quantity_Lost'].sum()
    total_value = df_lost['Value_Lost'].sum()
    
    print(f"Materials destroyed:  {len(df_lost)}")
    print(f"Total units lost:     {total_qty:,.0f}")
    print(f"Total value lost:     £{total_value:,.2f}")
    print("=" * 70)
    
    # Summary by material type
    type_summary = df_lost.groupby('Material_Type').agg({
        'Material': 'count',
        'Quantity_Lost': 'sum',
        'Value_Lost': 'sum'
    }).reset_index()
    type_summary.columns = ['Type', 'Materials', 'Qty_Lost', 'Value_Lost']
    print("\nBy Material Type:")
    for _, row in type_summary.iterrows():
        pct = (row['Value_Lost'] / total_value * 100) if total_value > 0 else 0
        print(f"  {row['Type']}: {row['Materials']} materials, {row['Qty_Lost']:,.0f} units, £{row['Value_Lost']:,.2f} ({pct:.1f}%)")
    
    print()
    display(lost_inventory)
else:
    print("No fire damage records found (SCN003).")

In [ ]:
# Chart: Lost Inventory
if len(df_lost) > 0:
    # Dynamic figure height
    fig_height = max(6, len(df_lost) * 0.5)
    fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))
    
    # Sort by value
    df_chart = df_lost.sort_values('Value_Lost', ascending=True)
    
    # Colors by material type
    colors = ['#d62728' if mt == 'FERT' else '#1f77b4' for mt in df_chart['Material_Type']]
    
    # Chart 1: Quantity Lost
    axes[0].barh(df_chart['Material'], df_chart['Quantity_Lost'], color=colors, height=0.7)
    axes[0].set_xlabel('Quantity Lost (Units)', fontsize=11)
    axes[0].set_title('Volume Lost', fontsize=12, fontweight='bold')
    axes[0].set_xlim(0, df_chart['Quantity_Lost'].max() * 1.15)
    
    # Chart 2: Value Lost
    axes[1].barh(df_chart['Material'], df_chart['Value_Lost'], color=colors, height=0.7)
    axes[1].set_xlabel('Value Lost (£)', fontsize=11)
    axes[1].set_title('Financial Loss', fontsize=12, fontweight='bold')
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
    axes[1].set_xlim(0, df_chart['Value_Lost'].max() * 1.15)
    
    fig.suptitle(f'Fire Damage: Lost Inventory at Plant {FIRE_PLANT}', fontsize=14, fontweight='bold')
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#d62728', label='Finished Goods (FERT)'),
                       Patch(facecolor='#1f77b4', label='Raw Materials (ROH)')]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.98))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

---
## 6. Product Shortfall Analysis

Comparing available inventory vs. customer demand to identify shortfalls caused by the fire.

In [ ]:
# Shortfall Analysis: Available Product vs Customer Orders
shortfall_analysis = spark.sql(f"""
    WITH fire_lost AS (
        -- Inventory destroyed in fire
        SELECT 
            MATNR,
            WERKS,
            SUM(CAST(MENGE AS DOUBLE)) as Qty_Lost
        FROM {CATALOG}.{SCHEMA}.matdoc
        WHERE MBLNR LIKE 'SCN003%'
        AND BWART = '551'
        GROUP BY MATNR, WERKS
    ),
    current_stock AS (
        -- Current available inventory (post-fire)
        SELECT 
            MATNR,
            WERKS,
            SUM(CAST(LABST AS DOUBLE)) as Available_Stock
        FROM {CATALOG}.{SCHEMA}.mard
        GROUP BY MATNR, WERKS
    ),
    delivered_orders AS (
        SELECT DISTINCT f.VBELN
        FROM {CATALOG}.{SCHEMA}.vbfa f
        INNER JOIN {CATALOG}.{SCHEMA}.likp d ON f.VBELN_N = d.VBELN
    ),
    open_demand AS (
        -- Open customer orders (demand) for next 3 months
        SELECT 
            p.MATNR,
            p.WERKS,
            SUM(CAST(p.KWMENG AS DOUBLE)) as Order_Demand,
            SUM(CAST(p.NETWR AS DOUBLE)) as Order_Value,
            COUNT(DISTINCT o.VBELN) as Order_Count
        FROM {CATALOG}.{SCHEMA}.vbak o
        INNER JOIN {CATALOG}.{SCHEMA}.vbap p ON o.VBELN = p.VBELN
        LEFT JOIN {CATALOG}.{SCHEMA}.vbep e ON p.VBELN = e.VBELN AND p.POSNR = e.POSNR
        WHERE o.VBELN NOT IN (SELECT VBELN FROM delivered_orders)
        AND e.EDATU >= '{FIRE_DATE_STR}'
        AND e.EDATU <= '{END_DATE_STR}'
        GROUP BY p.MATNR, p.WERKS
    )
    SELECT 
        COALESCE(d.MATNR, s.MATNR, f.MATNR) as Material,
        t.MAKTX as Material_Description,
        a.MTART as Material_Type,
        COALESCE(d.WERKS, s.WERKS, f.WERKS) as Plant,
        COALESCE(s.Available_Stock, 0) as Current_Stock,
        COALESCE(f.Qty_Lost, 0) as Fire_Loss,
        COALESCE(d.Order_Demand, 0) as Customer_Demand,
        COALESCE(d.Order_Value, 0) as Demand_Value,
        COALESCE(d.Order_Count, 0) as Orders,
        COALESCE(s.Available_Stock, 0) - COALESCE(d.Order_Demand, 0) as Stock_vs_Demand,
        CASE 
            WHEN COALESCE(s.Available_Stock, 0) < COALESCE(d.Order_Demand, 0) 
            THEN COALESCE(d.Order_Demand, 0) - COALESCE(s.Available_Stock, 0)
            ELSE 0 
        END as Shortfall_Qty,
        CASE WHEN f.MATNR IS NOT NULL THEN 'Yes' ELSE 'No' END as Fire_Affected
    FROM open_demand d
    FULL OUTER JOIN current_stock s ON d.MATNR = s.MATNR AND d.WERKS = s.WERKS
    FULL OUTER JOIN fire_lost f ON COALESCE(d.MATNR, s.MATNR) = f.MATNR AND COALESCE(d.WERKS, s.WERKS) = f.WERKS
    LEFT JOIN {CATALOG}.{SCHEMA}.makt t ON COALESCE(d.MATNR, s.MATNR, f.MATNR) = t.MATNR
    LEFT JOIN {CATALOG}.{SCHEMA}.mara a ON COALESCE(d.MATNR, s.MATNR, f.MATNR) = a.MATNR
    WHERE COALESCE(d.Order_Demand, 0) > 0 OR COALESCE(f.Qty_Lost, 0) > 0
    ORDER BY 
        CASE WHEN f.MATNR IS NOT NULL THEN 0 ELSE 1 END,
        CASE WHEN COALESCE(s.Available_Stock, 0) < COALESCE(d.Order_Demand, 0) THEN 0 ELSE 1 END,
        COALESCE(d.Order_Demand, 0) - COALESCE(s.Available_Stock, 0) DESC
""")

df_shortfall = shortfall_analysis.toPandas()

print("PRODUCT SHORTFALL ANALYSIS")
print("=" * 70)

# Filter to show shortfalls
shortfalls = df_shortfall[df_shortfall['Shortfall_Qty'] > 0]
fire_affected = df_shortfall[df_shortfall['Fire_Affected'] == 'Yes']

print(f"Total products with demand:        {len(df_shortfall)}")
print(f"Products with shortfall:           {len(shortfalls)}")
print(f"Fire-affected products:            {len(fire_affected)}")
print(f"Total shortfall quantity:          {shortfalls['Shortfall_Qty'].sum():,.0f} units")
print(f"Fire loss quantity:                {df_shortfall['Fire_Loss'].sum():,.0f} units")
print("=" * 70)
print()
display(shortfall_analysis)

In [ ]:
# Shortfall Chart - Fire affected products only
fire_products = df_shortfall[df_shortfall['Fire_Affected'] == 'Yes'].copy()

if len(fire_products) > 0:
    fig_height = max(6, len(fire_products) * 0.6)
    fig, ax = plt.subplots(figsize=(14, fig_height))
    
    # Sort by shortfall
    fire_products = fire_products.sort_values('Material')
    
    y_pos = range(len(fire_products))
    bar_height = 0.35
    
    # Current stock (green)
    bars1 = ax.barh([y - bar_height/2 for y in y_pos], fire_products['Current_Stock'], 
                    height=bar_height, label='Current Stock', color='#2ca02c', alpha=0.8)
    
    # Customer demand (blue)
    bars2 = ax.barh([y + bar_height/2 for y in y_pos], fire_products['Customer_Demand'], 
                    height=bar_height, label='Customer Demand', color='#1f77b4', alpha=0.8)
    
    # Fire loss indicator (red outline)
    for i, (idx, row) in enumerate(fire_products.iterrows()):
        if row['Fire_Loss'] > 0:
            ax.annotate(f"Lost: {row['Fire_Loss']:,.0f}", 
                       xy=(max(row['Current_Stock'], row['Customer_Demand']) + 100, i),
                       fontsize=9, color='#d62728', fontweight='bold')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(fire_products['Material'])
    ax.set_xlabel('Quantity (Units)', fontsize=11)
    ax.set_title(f'Fire-Affected Products: Stock vs Demand at Plant {FIRE_PLANT}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    
    # Add vertical line at 0
    ax.axvline(x=0, color='black', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table for fire-affected products
    print("\nFIRE-AFFECTED PRODUCTS SUMMARY:")
    print("-" * 90)
    print(f"{'Material':<12} {'Current':>12} {'Demand':>12} {'Fire Loss':>12} {'Shortfall':>12} {'Status':<15}")
    print("-" * 90)
    for _, row in fire_products.iterrows():
        status = 'SHORTFALL' if row['Shortfall_Qty'] > 0 else 'OK'
        print(f"{row['Material']:<12} {row['Current_Stock']:>12,.0f} {row['Customer_Demand']:>12,.0f} {row['Fire_Loss']:>12,.0f} {row['Shortfall_Qty']:>12,.0f} {status:<15}")
    print("-" * 90)
else:
    print("No fire-affected products with demand found.")

In [ ]:
# Executive Summary
print("\n" + "=" * 70)
print("FIRE SCENARIO - EXECUTIVE SUMMARY")
print("=" * 70)

# Lost inventory
total_lost_qty = df_lost['Quantity_Lost'].sum() if len(df_lost) > 0 else 0
total_lost_value = df_lost['Value_Lost'].sum() if len(df_lost) > 0 else 0

# Unfulfilled orders from fire location
unfulfilled_orders = len(df_fire_orders['Order_Number'].unique()) if len(df_fire_orders) > 0 else 0
unfulfilled_value = df_fire_orders['Line_Value'].sum() if len(df_fire_orders) > 0 else 0

# Shortfall
total_shortfall = shortfalls['Shortfall_Qty'].sum() if len(shortfalls) > 0 else 0

print(f"\n{'INVENTORY IMPACT':<40}")
print(f"  Materials destroyed:           {len(df_lost):>15,}")
print(f"  Units destroyed:               {total_lost_qty:>15,.0f}")
print(f"  Value destroyed:               £{total_lost_value:>14,.2f}")

print(f"\n{'CUSTOMER IMPACT':<40}")
print(f"  Orders at risk (fire plant):   {unfulfilled_orders:>15,}")
print(f"  Revenue at risk:               £{unfulfilled_value:>14,.2f}")

print(f"\n{'SUPPLY SHORTFALL':<40}")
print(f"  Products with shortfall:       {len(shortfalls):>15,}")
print(f"  Total shortfall (units):       {total_shortfall:>15,.0f}")

print("\n" + "=" * 70)